# This notebook does the below task 

Data preparation pipeline for Amazon product reviews, filtering the full dataset down to the top 30 most reviewed products and enriching it with product category labels.


## What it does, step by step:

Setup: Installs and imports libraries (DuckDB, pandas,BeautifulSoup, matplotlib), then connects to an in-memory DuckDB database pointing at alexa_reviews.parquet.

Schema inspection: Describes the dataset structure: 10 columns including asin, reviewerID, overall (star rating), reviewText, reviewTime, verified, and vote.

Identifies top 30 products: Queries the parquet file to find the 30 ASINs with the most reviews .

Category scraping: Defines a function that hits each product's Amazon page and extracts its category from the page title. Loops through all 30 ASINs with a 4-second delay to avoid rate limiting, then saves results to products_categories.csv. A few products returned None (blocked by Amazon) and were later filled in manually.

Builds a DuckDB view for the categories CSV and confirms all 30 ASINs have categories assigned. The final breakdown is dominated by Books (17 products), followed by Movies & TV (5), Audible (3), Electronics (2), Pet Supplies (2), and one each of Amazon Devices, Home & Kitchen, and Clothing.

Filters and joins: Extracts only reviews for the top 30 products from the full parquet file (829k rows), joins in the category labels, and saves the result as amazon_reviews_categories.csv.

Validation: Confirms the final dataset contains 829,621 rows with 11 columns (original 10 + category).


In [99]:
#install necessary libraries
!pip install duckdb

!pip install requests beautifulsoup4

!pip install matplotlib

python(77094) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


python(77096) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


python(77097) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [103]:
#Load Neccessary Libraries

import requests
from bs4 import BeautifulSoup
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import time
import random

In [101]:
# Establish the file path to the Parquet file

FILE = "alexa_reviews.parquet"

# Create a DuckDB connection
con = duckdb.connect(database=":memory:")


In [102]:
       
# view the data schema
con.execute(f"DESCRIBE SELECT * FROM '{FILE}' LIMIT 0").fetchdf()

,column_name,column_type,null,key,default,extra
0,asin,VARCHAR,YES,None,None,None
1,reviewerID,VARCHAR,YES,None,None,None
2,reviewerName,VARCHAR,YES,None,None,None
3,overall,DOUBLE,YES,None,None,None
4,summary,VARCHAR,YES,None,None,None
5,reviewText,VARCHAR,YES,None,None,None
6,unixReviewTime,BIGINT,YES,None,None,None
7,reviewTime,VARCHAR,YES,None,None,None
8,verified,BOOLEAN,YES,None,None,None
9,vote,VARCHAR,YES,None,None,None


### Identify Top 30 products

In [96]:
#Top 30 product
top30_products = con.execute(f"""SELECT asin, COUNT(*) AS reviews_per_product
FROM "{FILE}"
GROUP BY asin
ORDER BY reviews_per_product DESC
LIMIT 30;
""").fetchdf()

top30_products

,asin,reviews_per_product
0,038568231X,58150
1,0297859382,44956
2,0007420412,44381
3,0141353678,37783
4,0312577222,36620
5,0099911701,33676
6,B000X1MX7E,32495
7,0553418025,30297
8,B010OYASRG,28539
9,B00CYQP3AK,28304


### Create Category Extraction Function


In [104]:
def get_category(asin):
    # Construct the URL for the product page using the ASIN  
    url = f"https://www.amazon.com/dp/{asin}"

    # Set a user-agent header to mimic a browser request 
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9"
    }
    
    
    try:
        # Send a GET request to the product page
        response = requests.get(url, headers=headers, timeout=10)
        # Check if the request was successful
        if response.status_code != 200:
            return None
        
        # Parse the HTML content
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Extract category breadcrumb links
        if soup.title:
            title = soup.title.text.strip()

            if ":" in title:
                return title.split(":")[-1].strip()

        return None

    except:
        return None

In [105]:

categories = []

for asin in top30_products["asin"]:

    category = get_category(asin)

    categories.append({
        "asin": asin,
        "category": category
    })

    print(f"{asin} : {category}")

    # Delay to avoid Amazon blocking
    time.sleep(random.uniform(8,15))
    
#create a DataFrame from the categories list
category_df = pd.DataFrame(categories)

#save the DataFrame to a CSV file
#category_df.to_csv("products_categories.csv", index=False)


038568231X : Books
0297859382 : Books
0007420412 : Books
0141353678 : Books
0312577222 : Books
0099911701 : Books
B000X1MX7E : Audible Books & Originals
0553418025 : Books
B010OYASRG : Electronics
B00CYQP3AK : Amazon Devices & Accessories
0007548672 : Books
B00FLYWNYQ : None
B000W5QSYA : None
0316055433 : Books
B00YSG2ZPA : Movies & TV
B00006CXSS : Movies & TV
B000WGWQG8 : Movies & TV
0439023521 : Books
8184776217 : Books
0545582881 : Books
B017WJ5PR4 : Audible Books & Originals
B017V4IPPO : Audible Books & Originals
0385537859 : Books
B00AQVMZKQ : Movies & TV
B01BHTSIOC : Movies & TV
B00L0YLRUW : Electronics
0143125478 : Books
006195070X : Books
B000YXC2LI : None
B000OX89XI : Pet Supplies


In [107]:
#create a dictionary for the missing categories.
manually_searched_categories = {
    "B00FLYWNYQ": "Home & Kitchen",
    "B000W5QSYA": "Pet Supplies",
    "B000YXC2LI": "Clothing, Shoes & Jewelry"

}

category_df["category"] = category_df["category"].fillna(
    category_df["asin"].map(manually_searched_categories)
)

In [ ]:
#Verify Everything Is Filled
category_df

,asin,category
0,038568231X,Books
1,0297859382,Books
2,0007420412,Books
3,0141353678,Books
4,0312577222,Books
5,0099911701,Books
6,B000X1MX7E,Audible Books & Originals
7,0553418025,Books
8,B010OYASRG,Electronics
9,B00CYQP3AK,Amazon Devices & Accessories


In [109]:
#Save the Final Clean CSV
category_df.to_csv("products_categories.csv", index=False)

### Product Category Extraction for Top Reviewed Products

To better understand the types of products generating the highest review activity, the top 30 most reviewed products were identified using their ASIN (Amazon Standard Identification Number). Each ASIN was then used to retrieve the corresponding product category by querying the Amazon product page and extracting the category information from the page title.

The results show that a large proportion of the most reviewed products belong to media-related categories such as **Books**, **Audible Books & Originals**, and **Movies & TV**. This suggests that entertainment-related products tend to generate a higher volume of user engagement and review activity within the dataset. A smaller number of products were also found in categories such as **Electronics**, **Amazon Devices & Accessories**, and **Pet Supplies**, indicating that highly reviewed items span multiple product domains.

some products returned **None** as their category. This occurs because Amazon may block automated requests, return incomplete page content, or load a generic page that does not contain product metadata. As a result, the scraper is unable to extract the category information for those specific ASINs. This is a common limitation when collecting product information through automated web requests and does not necessarily indicate missing product data.

Despite a small number of missing categories, the extracted results still provide useful insight into the types of products that dominate the most reviewed items in the dataset.

In order to solve this the product which had none will be seeacrhed for manually and inputed into the dataset

In [110]:

# create a view in DuckDB to read the CSV file
con.execute("""
CREATE OR REPLACE TABLE products_categories AS
SELECT
    TRIM(asin) AS asin,
    TRIM(category) AS category
FROM read_csv_auto('products_categories.csv')
""").fetchdf()


,Count
0,30


### Check That ASINs Exist in the Review Dataset

In [112]:
con.execute(f"""
SELECT COUNT(*)
FROM "{FILE}"
WHERE asin = '038568231X'
""").fetchdf()

,count_star()
0,58150


### Create a Table for the Top 30 Products

In [113]:
# Create a view for the top 30 products
con.register("top30_products", top30_products).fetchdf()


In [114]:
#confirm the view is working
con.execute("""
SELECT *
FROM top30_products
""").fetchdf()

,asin,reviews_per_product
0,038568231X,58150
1,0297859382,44956
2,0007420412,44381
3,0141353678,37783
4,0312577222,36620
5,0099911701,33676
6,B000X1MX7E,32495
7,0553418025,30297
8,B010OYASRG,28539
9,B00CYQP3AK,28304


### Extract Reviews Only for Those Products

In [115]:
top30_reviews = con.execute(f"""
SELECT
    r.*
FROM "{FILE}" r
JOIN top30_products t
ON r.asin = t.asin
""").fetchdf()

top30_reviews

,asin,reviewerID,reviewerName,overall,summary,reviewText,unixReviewTime,reviewTime,verified,vote
0,B00CYQP3AK,A3UXVNOFC8TJPT,Bkonasek,5.0,Great product,I received this as a birthday present. I am do...,1382140800,"10 19, 2013",False,2
1,B00CYQP3AK,AC2VR9U3NN2UD,tikitoo,2.0,Battery Issues and Light bleed - 2nd time Not ...,Battery will not charge to 100%. When I receiv...,1382140800,"10 19, 2013",True,22
2,B00CYQP3AK,A14D0LJ1YRLLCI,gabbie03,1.0,Missing promised features. Not ready for prime...,Buyer Beware. these units are being shipped ri...,1382140800,"10 19, 2013",True,54
3,B00CYQP3AK,A3FVF1JIOA5GEK,MN_Ranger,5.0,Amazing Performance Worthy of Upgrade,"Netflix Issue:\nWhen I first got the KFHDX, I ...",1382054400,"10 18, 2013",True,577
4,B00CYQP3AK,A3KP101J8H7EWY,L.D.K,5.0,This is by far the best tablet for the price,"I have owned many versions of the Kindle, and ...",1382054400,"10 18, 2013",True,8
...,...,...,...,...,...,...,...,...,...,...
829616,0141353678,A2MRLRA5WLJ24E,Gillie,5.0,Beautiful.,"I sat down to read this last night, not really...",1326758400,"01 17, 2012",False,None
829617,0141353678,A2ZQYF0SMHCJ49,AllieCat,5.0,You will read it again and again!!!,I was alternating between crying my eyes out a...,1326758400,"01 17, 2012",True,2
829618,0141353678,A1WD22IBEAFMT,Sean O,5.0,Just amazing,I just finished this book and I have to say it...,1326758400,"01 17, 2012",True,None
829619,0141353678,A2WKXJ5DDP375N,jcavvy,5.0,Beyond amazing - a must read!,"I strongly believe that a book must feel real,...",1326758400,"01 17, 2012",False,None


### Join and save Categories With Reviews

In [116]:
top30_product_reviews = con.execute(f"""
SELECT
    r.asin,
    c.category,
    r.reviewerID,
    r.reviewerName,
    r.overall,
    r.summary,
    r.reviewText,
    r.unixReviewTime,
    r.reviewTime,
    r.verified,
    r.vote
FROM "{FILE}" r
JOIN top30_products t
ON r.asin = t.asin
LEFT JOIN products_categories c
ON r.asin = c.asin
""").fetchdf()
top30_product_reviews.to_csv("amazon_reviews_categories.csv", index=False)


In [117]:
#view the enriched dataset
con.execute("""
SELECT *
FROM read_csv_auto('amazon_reviews_categories.csv')
LIMIT 10
""").fetchdf()

,asin,category,reviewerID,reviewerName,overall,summary,reviewText,unixReviewTime,reviewTime,verified,vote
0,B00CYQP3AK,Amazon Devices & Accessories,A3UXVNOFC8TJPT,Bkonasek,5.0,Great product,I received this as a birthday present. I am do...,1382140800,"10 19, 2013",False,2
1,B00CYQP3AK,Amazon Devices & Accessories,AC2VR9U3NN2UD,tikitoo,2.0,Battery Issues and Light bleed - 2nd time Not ...,Battery will not charge to 100%. When I receiv...,1382140800,"10 19, 2013",True,22
2,B00CYQP3AK,Amazon Devices & Accessories,A14D0LJ1YRLLCI,gabbie03,1.0,Missing promised features. Not ready for prime...,Buyer Beware. these units are being shipped ri...,1382140800,"10 19, 2013",True,54
3,B00CYQP3AK,Amazon Devices & Accessories,A3FVF1JIOA5GEK,MN_Ranger,5.0,Amazing Performance Worthy of Upgrade,"Netflix Issue:\nWhen I first got the KFHDX, I ...",1382054400,"10 18, 2013",True,577
4,B00CYQP3AK,Amazon Devices & Accessories,A3KP101J8H7EWY,L.D.K,5.0,This is by far the best tablet for the price,"I have owned many versions of the Kindle, and ...",1382054400,"10 18, 2013",True,8
5,B00CYQP3AK,Amazon Devices & Accessories,AFM7AQ69FH3YR,Amazon Customer,4.0,Very Good Tablet....But No GPS and Limited Apps,"SUMMARY - Overall, I really like the Kindle Fi...",1382054400,"10 18, 2013",True,94
6,B00CYQP3AK,Amazon Devices & Accessories,A3HHCD6UU62VNH,Rick G,5.0,An awesome tablet for the money,"We received the Kindle Fire HDX 7"" about a wee...",1382054400,"10 18, 2013",True,16
7,B00CYQP3AK,Amazon Devices & Accessories,A2AWCP5B215EAL,David,5.0,I love the new Kindle HDX,"I have a first generation Kindle Fire, it did ...",1382054400,"10 18, 2013",True,25
8,B00CYQP3AK,Amazon Devices & Accessories,A2E9C2SELN8Q68,BarbZ,4.0,"Gorgeous device, but some things you should kn...",This device is a breathtaking movie watching d...,1382054400,"10 18, 2013",True,376
9,B00CYQP3AK,Amazon Devices & Accessories,A10PEXB6XAQ5XF,Michael Gallagher,4.0,Nice Tablet with Sharp Display,To sum up what I will tell you about in the de...,1382054400,"10 18, 2013",True,"4,317"


### Check the total number of rows in the dataset

In [119]:
DATA_FILE = "amazon_reviews_categories.csv"

In [120]:
#Check the total number of rows in the dataset
row_count = con.execute(f"""
            SELECT COUNT(*) AS total_rows
            FROM '{DATA_FILE}'
            """).fetchdf()
row_count

,total_rows
0,829621


From 233 million reviews in the original dataset, the data has been have filtered down to 829,621 thousand reviews for the top 30 most reviewed products, enriched with category labels.